# Trabajo Final Deep Learning — Módulo LLM / Chatbot RAG v9

**Versión alineada con la app final Angular + Node + Ollama e integración CNN/MLP.**

Este notebook documenta y reproduce el componente **LLM / Chatbot RAG** del proyecto final. A diferencia de versiones anteriores, esta versión ya considera que la aplicación integra:

- **CNN visual** desde `public/data/cnn-scores.json`.
- **MLP tabular** desde `public/data/mlp-scores.json`.
- **Score textual / reseñas** desde `public/data/review-sentiment.json`.
- **Fusión tardía multimodal** con pesos iguales.
- **Chatbot RAG local** conectado a Ollama con `llama3.1:8b`.

El notebook sirve como evidencia técnica para explicar la parte del chatbot y su conexión con la decisión multimodal de la app.

## 0. Evaluación previa de la app final


```text
Angular selecciona alojamiento
→ envía listingId + pregunta + contexto multimodal
→ Node/RAG recupera ficha + reseñas del mismo ID Airbnb
→ detecta intención
→ filtra evidencias útiles
→ construye prompt para Ollama
→ devuelve respuesta + trazabilidad
→ Angular muestra respuesta, evidencias y scores CNN/MLP/Reseñas/Fusión
```

**Fortalezas verificadas de la app:**

- Usa `POST /api/rag-chat` y `listingId`, por lo que evita mezclar reseñas de distintos alojamientos.
- El backend reporta `mode: ollama-rag` y `model: llama3.1:8b` cuando Ollama responde.
- Existe separación entre evidencias usadas y metadatos visuales de recuperación.
- La app ya carga scores de **CNN**, **MLP**, **reseñas/NLP** y calcula **fusión tardía**.
- El botón de **Resumen comercial** envía contexto multimodal al backend para explicar la decisión.
- El botón de exportación guarda decisión + conversación + evidencias en JSON.

**Observaciones a cuidar en la exposición:**

- El fallback extractivo existe solo para que la demo no se caiga si Ollama falla. Para la presentación, se debe ejecutar Ollama.
- La métrica de recuperación del RAG no debe interpretarse como rating ni como score del alojamiento.
- El notebook no reemplaza la app; documenta el componente LLM/RAG y demuestra cómo se conecta con CNN/MLP.

## 1. Instalación mínima

Ejecuta esta celda solo si tu entorno no tiene las librerías. Se evita instalar `sentence-transformers` por defecto porque en Windows puede causar errores de DLL; el recuperador de este notebook usa TF-IDF y reglas de intención estables.

In [ ]:
# Instalación mínima estable
# %pip -q install pandas numpy scikit-learn requests ipywidgets

## 2. Importación de librerías y configuración

In [1]:
import json
import math
import os
import re
import textwrap
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown

# Configuración principal
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "llama3.1:8b")
LLM_PROVIDER = "ollama"  # opciones: "ollama" o "none"
TOP_K = 5

# Si ejecutas el notebook fuera de la carpeta del proyecto, puedes fijar manualmente la ruta:
PROJECT_ROOT_MANUAL = r"C:\Users\andre\OneDrive\Documentos\Deep Learning\TF_DL_Grupo4"

## 3. Localización del proyecto y carga de datos integrados

Esta versión trabaja directamente con los JSON usados por la app, porque ya integran el dataset limpio y las salidas de los modelos CNN/MLP/NLP.

In [2]:
def find_project_root() -> Path:
    """Busca la raíz del proyecto Angular a partir del directorio actual o una ruta manual."""
    candidates: List[Path] = []

    if PROJECT_ROOT_MANUAL:
        candidates.append(Path(PROJECT_ROOT_MANUAL))

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidates.extend([
            base,
            base / "TF_DL_Grupo4",
            base / "TF_DL_Grupo4-main",
            base / "TF_DL_Grupo4-main_finale",
        ])

    # Ruta útil solo para esta revisión en sandbox; en tu PC normalmente no existe.
    candidates.append(Path("/mnt/data/TF_DL_Grupo4_inspect/TF_DL_Grupo4-main"))

    for candidate in candidates:
        if (candidate / "public/data/listings.json").exists() and (candidate / "server/rag-server.mjs").exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. Coloca este notebook dentro del proyecto o define PROJECT_ROOT_MANUAL."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "public" / "data"

LISTINGS_PATH = DATA_DIR / "listings.json"
CNN_PATH = DATA_DIR / "cnn-scores.json"
MLP_PATH = DATA_DIR / "mlp-scores.json"
REVIEW_SENTIMENT_PATH = DATA_DIR / "review-sentiment.json"
IMAGE_MANIFEST_PATH = DATA_DIR / "image-manifest.json"

print("Proyecto detectado:", PROJECT_ROOT)
print("Datos:", DATA_DIR)

Proyecto detectado: C:\Users\andre\OneDrive\Documentos\Deep Learning\TF_DL_Grupo4
Datos: C:\Users\andre\OneDrive\Documentos\Deep Learning\TF_DL_Grupo4\public\data


In [3]:
def read_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

listings_data = read_json(LISTINGS_PATH)
cnn_scores = read_json(CNN_PATH)
mlp_scores = read_json(MLP_PATH)
review_sentiment = read_json(REVIEW_SENTIMENT_PATH) if REVIEW_SENTIMENT_PATH.exists() else {"listings": {}}
image_manifest = read_json(IMAGE_MANIFEST_PATH) if IMAGE_MANIFEST_PATH.exists() else {"listings": {}}

listings = listings_data["listings"]
listing_by_id = {str(item["id"]): item for item in listings}

print("Listados:", len(listings))
print("Reseñas cruzadas:", sum(len(item.get("reviews", [])) for item in listings))
print("CNN scores:", len(cnn_scores.get("listings", {})))
print("MLP scores:", len(mlp_scores.get("listings", {})))
print("Review sentiment/NLP:", len(review_sentiment.get("listings", {})))
print("Manifest imágenes:", len(image_manifest.get("listings", {})))

Listados: 52
Reseñas cruzadas: 1592
CNN scores: 52
MLP scores: 52
Review sentiment/NLP: 172
Manifest imágenes: 52


## 4. Auditoría rápida de integración de la app

Esta celda verifica que los artefactos que consume Angular están alineados por `ID Airbnb`.

In [4]:
def audit_app_data() -> pd.DataFrame:
    ids = set(listing_by_id.keys())
    rows = []
    artifacts = {
        "CNN": cnn_scores.get("listings", {}),
        "MLP": mlp_scores.get("listings", {}),
        "Review sentiment/NLP": review_sentiment.get("listings", {}),
        "Image manifest": image_manifest.get("listings", {}),
    }
    for name, data in artifacts.items():
        artifact_ids = set(map(str, data.keys()))
        rows.append({
            "artefacto": name,
            "registros": len(artifact_ids),
            "ids_app_cubiertos": len(ids & artifact_ids),
            "faltantes_en_app": len(ids - artifact_ids),
            "extras_fuera_app": len(artifact_ids - ids),
        })
    return pd.DataFrame(rows)

audit_df = audit_app_data()
display(audit_df)

meta = listings_data.get("meta", {})
print("Fusión configurada en app:", meta.get("fusionWeights"))
print("Umbrales de decisión:", meta.get("decisionThresholds"))
print("Tópicos principales:", meta.get("topReviewTopics"))

,artefacto,registros,ids_app_cubiertos,faltantes_en_app,extras_fuera_app
0,CNN,52,52,0,0
1,MLP,52,52,0,0
2,Review sentiment/NLP,172,52,0,120
3,Image manifest,52,52,0,0


Fusión configurada en app: {'vision': 0.3333333333333333, 'tabular': 0.3333333333333333, 'reviews': 0.3333333333333333}
Umbrales de decisión: {'recommended': 75, 'review': 50, 'notRecommendedBelow': 50}
Tópicos principales: [['ubicación', 49], ['limpieza', 48], ['anfitrión', 48], ['comodidad', 47], ['precio', 37]]


## 5. Catálogo de alojamientos

La app ordena los alojamientos por puntaje de decisión. En el notebook se muestra una vista resumida para seleccionar un `ID Airbnb` de prueba.

In [5]:
def label_from_score(score: float, high=75, medium=50, high_label="Alta", mid_label="Media", low_label="Baja") -> str:
    if score >= high:
        return high_label
    if score >= medium:
        return mid_label
    return low_label

def get_model_scores_for_listing(listing_id: str) -> Dict[str, Any]:
    listing_id = str(listing_id)
    cnn = cnn_scores.get("listings", {}).get(listing_id)
    mlp = mlp_scores.get("listings", {}).get(listing_id)
    rev = review_sentiment.get("listings", {}).get(listing_id)
    weights = listings_data.get("meta", {}).get("fusionWeights", {"vision": 1/3, "tabular": 1/3, "reviews": 1/3})

    vision_score = float(cnn["score"]) if cnn else 50.0
    vision_conf = float(cnn.get("confidence", cnn_scores.get("meta", {}).get("confidence", 50))) if cnn else 15.0
    vision_label = cnn.get("label") if cnn else "Sin foto"

    mlp_score = float(mlp["score"]) if mlp else np.nan
    mlp_conf = float(mlp_scores.get("meta", {}).get("confidence", 70)) if mlp else 0.0
    mlp_label = label_from_score(mlp_score) if mlp else "No disponible"

    rev_score = float(rev["score"]) if rev else 50.0
    rev_conf = float(rev.get("confidence", 25)) if rev else 25.0
    rev_label = label_from_score(rev_score) if rev else "Sin evidencia"

    fusion_score = (
        vision_score * float(weights.get("vision", 1/3)) +
        mlp_score * float(weights.get("tabular", 1/3)) +
        rev_score * float(weights.get("reviews", 1/3))
    )
    fusion_conf = (
        vision_conf * float(weights.get("vision", 1/3)) +
        mlp_conf * float(weights.get("tabular", 1/3)) +
        rev_conf * float(weights.get("reviews", 1/3))
    )
    fusion_label = "Recomendado" if fusion_score >= 75 else "Revisar" if fusion_score >= 50 else "No recomendado"

    return {
        "vision": {"score": round(vision_score, 1), "label": vision_label, "confidence": round(vision_conf, 1)},
        "tabular": {"score": round(mlp_score, 1), "label": mlp_label, "confidence": round(mlp_conf, 1)},
        "reviews": {"score": round(rev_score, 1), "label": rev_label, "confidence": round(rev_conf, 1)},
        "fusion": {"score": round(fusion_score, 1), "label": fusion_label, "confidence": round(fusion_conf, 1), "weights": weights},
    }

def build_catalog() -> pd.DataFrame:
    rows = []
    for listing in listings:
        scores = get_model_scores_for_listing(str(listing["id"]))
        rows.append({
            "ID Airbnb": str(listing["id"]),
            "Título": listing.get("title", ""),
            "Host": listing.get("host", ""),
            "Precio": listing.get("price"),
            "Rating": listing.get("rating"),
            "Reseñas": listing.get("reviewCountMatched", len(listing.get("reviews", []))),
            "CNN": scores["vision"]["score"],
            "MLP": scores["tabular"]["score"],
            "Reviews/NLP": scores["reviews"]["score"],
            "Fusión": scores["fusion"]["score"],
            "Decisión": scores["fusion"]["label"],
        })
    return pd.DataFrame(rows).sort_values("Fusión", ascending=False)

catalog_df = build_catalog()
display(catalog_df.head(10))

,ID Airbnb,Título,Host,Precio,Rating,Reseñas,CNN,MLP,Reviews/NLP,Fusión,Decisión
9,1207414356654599414,Exclusivo Apt 1BR con Netflix Gratis | 507,Diana Rosa,140.0,4.96,24,68.8,96.7,97.9,87.8,Recomendado
42,1194885970937542129,Apartamento en Barranco-Lima,Raúl,130.0,4.88,3,65.7,97.1,100.0,87.6,Recomendado
22,1242307686892251783,Apartamento de lujo |Jacuzzi|Piscina|cerca del...,Sandra,180.0,4.84,4,62.6,97.1,100.0,86.6,Recomendado
11,1089836481621094489,"Luxury Loft Barranco 1111 Pool, Gym, Cowork, P...",Francisco,145.0,4.92,50,65.6,97.3,95.0,86.0,Recomendado
24,1202250552806361613,Apartamento de lujo con vistas al mar y a la c...,Rocko De El Depa De Rocko,207.0,4.94,16,60.3,96.9,100.0,85.7,Recomendado
10,1191330136307274968,"Piso de 2 dormitorios con acceso a la piscina,...",Nati,142.0,4.81,13,59.7,96.5,100.0,85.4,Recomendado
13,846313424333805576,*Barranco: Edificio de estreno | Ideal Parejas*,Mafer,150.0,4.86,46,65.2,97.1,89.1,83.8,Recomendado
18,1247380646917486240,Departamento de lujo con piscina/gimnasio/cowo...,Rocko De El Depa De Rocko,157.0,4.91,4,64.7,96.9,87.5,83.0,Recomendado
19,800208014785752524,Increíble apartamento nuevo Wifi+Gimnasio+Pisc...,Lesly,158.0,4.92,27,56.8,96.9,94.4,82.7,Recomendado
3,899379506224853216,Apt en Barranco con bicicletas-Allin Samay,Aldo Bruno,119.0,4.94,32,55.8,96.9,95.3,82.7,Recomendado


## 6. Utilidades de texto e intención

Estas funciones reproducen la lógica conceptual del backend RAG: normalización, detección de intención y expansión de términos de búsqueda.

In [6]:
QUERY_STOP_WORDS = {
    "tiene", "tener", "hay", "esta", "este", "para", "sobre", "como", "que", "cual", "cuales",
    "donde", "cuando", "opinan", "dicen", "huespedes", "huéspedes", "departamento", "alojamiento"
}

TOPIC_KEYWORDS = {
    "amenidades": ["amenidad", "amenidades", "servicio", "servicios", "piscina", "pool", "jacuzzi", "gimnasio", "gym", "coworking", "cowork", "wifi", "wi fi", "wi-fi", "internet", "aire acondicionado", "cocina", "lavadora", "secadora", "estacionamiento", "balcon", "balcón", "televisor", "smart tv"],
    "limpieza": ["limpieza", "limpio", "limpia", "impecable", "ordenado", "aseado", "higiene"],
    "ubicacion": ["ubicacion", "ubicación", "zona", "barranco", "malecon", "malecón", "cerca", "cercania", "cercanía", "restaurantes", "cafeterias", "cafeterías", "tranquilo", "acceso"],
    "anfitrion": ["anfitrion", "anfitrión", "host", "amable", "respuesta", "respuestas", "atento", "atenta", "atencion", "atención", "comunicacion", "comunicación", "servicial", "resolver", "rapida", "rápida"],
    "comodidad": ["cama", "comodo", "cómodo", "acogedor", "descansar", "tranquilo", "espacio"],
    "precio": ["precio", "calidad", "valor", "razonable", "caro", "barato", "recomendado", "cumple"],
    "fotos": ["foto", "fotos", "igual", "publicadas", "moderno", "vista", "vistas"],
    "remoto": ["wifi", "internet", "trabajo", "remoto", "cowork", "coworking", "computador", "ordenador"],
    "quejas": ["queja", "quejas", "problema", "problemas", "malo", "mala", "sucio", "sucia", "ruido", "difícil", "dificil", "falta", "fallo", "privacidad"],
    "mejoras": ["mejorar", "mejora", "mejoras", "debilidad", "debilidades", "aspectos negativos", "negativo", "critica", "crítica", "criticas", "críticas"],
    "positivo": ["positivo", "positivos", "aspecto positivo", "aspectos positivos", "fortaleza", "fortalezas", "recomienda", "recomendado", "excelente", "bueno", "buena", "bonito", "agradable"],
    "sentimiento": ["sentimiento", "sentimientos", "polaridad", "positivo", "positiva", "negativo", "negativa", "neutral", "opinion", "opiniones", "satisfaccion", "emoción", "emocion"],
    "comercial": ["conviene", "administrar", "inversion", "inversión", "rentable", "negocio", "decision", "decisión", "comercial", "resumen comercial"],
    "capacidad": ["capacidad", "persona", "personas", "pareja", "parejas", "familia", "familias", "grupo", "grupos", "huesped", "huespedes", "huésped", "huéspedes", "cama", "camas", "habitacion", "habitación", "bano", "baño", "comodo", "cómodo", "equipado", "preparado", "estancia", "estadía"],
}

def normalize(text: Any) -> str:
    text = str(text or "").lower()
    text = text.replace("wi fi", "wifi").replace("wi-fi", "wifi")
    text = text.encode("utf-8", "ignore").decode("utf-8")
    text = re.sub(r"[\u0300-\u036f]", "", text)
    import unicodedata
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    text = re.sub(r"[^a-z0-9ñ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: Any) -> List[str]:
    return [t for t in normalize(text).split() if len(t) > 2]

def includes_term(text: str, term: str) -> bool:
    text_n = normalize(text)
    term_n = normalize(term)
    if not term_n:
        return False
    if " " in term_n:
        return term_n in text_n
    return re.search(rf"(^|\s){re.escape(term_n)}(\s|$)", text_n) is not None

def unique(values: List[str]) -> List[str]:
    seen = set()
    out = []
    for v in values:
        if v and v not in seen:
            seen.add(v)
            out.append(v)
    return out

def clean_review_text(text: Any) -> str:
    text = str(text or "")
    text = re.sub(r"<br\s*/?>", ". ", text, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def informative_review_text(text: Any) -> bool:
    clean = clean_review_text(text)
    norm = normalize(clean)
    if len(clean) < 35:
        return False
    banned = ["las resenas de los huespedes mencionan", "resenas de los huespedes"]
    return not any(phrase in norm for phrase in banned)

In [7]:
def detect_intent(question: str) -> Dict[str, Any]:
    qn = normalize(question)
    categories = set()
    for topic, keywords in TOPIC_KEYWORDS.items():
        if includes_term(qn, topic) or any(includes_term(qn, kw) for kw in keywords):
            categories.add(topic)

    if any(phrase in qn for phrase in ["que opinan", "dicen los huespedes", "experiencia"]):
        categories.add("experiencia")

    explicit_capacity = any(includes_term(qn, t) for t in ["capacidad", "cuantas", "cuantos", "persona", "personas", "familia", "grupo", "cama", "habitacion", "bano"])
    has_specific_non_capacity = any(c in categories for c in ["amenidades", "limpieza", "ubicacion", "anfitrion", "precio", "quejas", "mejoras", "remoto", "fotos", "positivo", "sentimiento"])
    if "capacidad" in categories and has_specific_non_capacity and not explicit_capacity:
        categories.remove("capacidad")

    if not categories:
        categories.add("general")

    amenity_terms = []
    for category in ["amenidades", "remoto"]:
        for kw in TOPIC_KEYWORDS[category]:
            if normalize(kw) not in ["amenidad", "amenidades", "servicio", "servicios"] and includes_term(qn, kw):
                amenity_terms.append(normalize(kw))

    return {
        "categories": sorted(categories),
        "asks_about_improvements": "mejoras" in categories,
        "asks_about_complaints": "quejas" in categories,
        "asks_about_capacity": "capacidad" in categories,
        "asks_commercial_decision": "comercial" in categories,
        "asks_about_amenities": "amenidades" in categories or "remoto" in categories,
        "asks_about_sentiment": "sentimiento" in categories,
        "amenity_terms": unique(amenity_terms),
    }

def expanded_question_tokens(question: str) -> List[str]:
    base = [t for t in tokenize(question) if t not in QUERY_STOP_WORDS]
    expanded = set(base)
    qn = normalize(question)
    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(includes_term(qn, kw) for kw in keywords):
            expanded.add(topic)
            for kw in keywords:
                if topic in ["amenidades", "remoto"]:
                    if includes_term(qn, kw):
                        expanded.add(normalize(kw))
                else:
                    expanded.add(normalize(kw))
    return [t for t in expanded if t]

def intent_keywords(intent: Dict[str, Any]) -> List[str]:
    words = []
    for cat in intent["categories"]:
        if cat in ["amenidades", "remoto"] and intent.get("amenity_terms"):
            words.extend(intent["amenity_terms"])
        else:
            words.extend(TOPIC_KEYWORDS.get(cat, []))
    return unique([normalize(w) for w in words if w])

## 7. Selección de ficha, reseñas y evidencia útil

La regla central es: **no enviar al LLM toda la recuperación cruda**, sino solo la evidencia útil para la intención de la pregunta.

In [8]:
def extract_listing_facts(listing: Dict[str, Any]) -> List[Dict[str, str]]:
    facts = []
    summary = listing.get("summary", "")
    for part in [p.strip() for p in summary.split("-") if p.strip()]:
        n = normalize(part)
        if "huesped" in n:
            facts.append({"label": "Capacidad", "value": part, "source": "Resumen de la propiedad"})
        elif "habitacion" in n:
            facts.append({"label": "Habitaciones", "value": part, "source": "Resumen de la propiedad"})
        elif "cama" in n:
            facts.append({"label": "Camas", "value": part, "source": "Resumen de la propiedad"})
        elif "bano" in n:
            facts.append({"label": "Baños", "value": part, "source": "Resumen de la propiedad"})

    facts.extend([
        {"label": "Precio", "value": f"S/ {round(float(listing.get('price', 0)))} por noche", "source": "Hoja Principal"},
        {"label": "Rating", "value": f"{float(listing.get('rating', 0)):.2f} / 5", "source": "Hoja Principal"},
        {"label": "Host", "value": str(listing.get("host", "No indicado")), "source": "Hoja Principal"},
        {"label": "Distrito", "value": str(listing.get("district", "No indicado")), "source": "Hoja Principal"},
        {"label": "Amenidades", "value": f"{listing.get('amenities', 'No indicado')}", "source": "Hoja Principal"},
        {"label": "Superhost", "value": "Sí" if listing.get("superhost") else "No", "source": "Hoja Principal"},
        {"label": "Tiempo como host", "value": f"{listing.get('hostYears', 'No indicado')} años", "source": "Hoja Principal"},
        {"label": "Reconocimiento", "value": str(listing.get("recognition", "No indicado")), "source": "Hoja Principal"},
    ])
    return facts

def snippet_around_terms(text: str, terms: List[str], max_len: int = 190) -> str:
    clean = clean_review_text(text)
    if not clean:
        return ""
    terms_n = unique([normalize(t) for t in terms if t])
    hit_positions = []
    for term in terms_n:
        m = re.search(rf"\b{re.escape(term)}\b", normalize(clean))
        if m:
            # approximate original position by simple search on normalized words
            idx = clean.lower().find(term)
            hit_positions.append(idx if idx >= 0 else 0)
    if not hit_positions:
        return clean if len(clean) <= max_len else clean[:max_len].strip() + "..."
    hit = max(0, min(hit_positions))
    start = max(0, hit - int(max_len * 0.35))
    end = min(len(clean), start + max_len)
    out = clean[start:end].strip()
    return ("..." if start > 0 else "") + out + ("..." if end < len(clean) else "")

def amenity_terms_from_intent(intent: Dict[str, Any]) -> List[str]:
    if intent.get("amenity_terms"):
        return intent["amenity_terms"]
    terms = []
    for cat in ["amenidades", "remoto"]:
        if cat in intent["categories"]:
            terms.extend(TOPIC_KEYWORDS[cat])
    return unique([normalize(t) for t in terms if normalize(t) not in ["amenidad", "amenidades", "servicio", "servicios"]])

def amenity_fact_text(description: str, intent: Dict[str, Any]) -> str:
    terms = amenity_terms_from_intent(intent)
    present = [t for t in terms if includes_term(description, t)]
    if present:
        return f"Incluye {', '.join(present[:4])}."
    return snippet_around_terms(description, terms, 170)

def location_fact_text(listing: Dict[str, Any], description: str) -> str:
    parts = []
    if listing.get("district"):
        parts.append(f"Alojamiento ubicado en {listing['district']}.")
    snippet = snippet_around_terms(description, ["ubicacion", "cerca", "zona", "restaurante", "cafeteria", "malecon", "miraflores"], 170)
    if any(includes_term(snippet, t) for t in ["cerca", "zona", "restaurante", "cafeteria", "malecon", "miraflores"]):
        parts.append(snippet)
    return " ".join(unique(parts))

def select_facts_for_intent(listing: Dict[str, Any], all_facts: List[Dict[str, str]], intent: Dict[str, Any]) -> List[Dict[str, str]]:
    labels = set()
    cats = set(intent["categories"])
    if "capacidad" in cats:
        labels.update(["Capacidad", "Habitaciones", "Camas", "Baños"])
    if "precio" in cats:
        labels.update(["Precio", "Rating", "Distrito"])
    if "ubicacion" in cats:
        labels.update(["Distrito"])
    if "anfitrion" in cats:
        labels.update(["Host", "Superhost", "Tiempo como host"])
    if "mejoras" in cats:
        labels.update(["Host"])
    if intent["asks_commercial_decision"] or "general" in cats:
        labels.update(["Capacidad", "Habitaciones", "Camas", "Baños", "Precio", "Rating", "Distrito", "Host", "Superhost"])

    selected = [f for f in all_facts if f["label"] in labels]
    description = " ".join([str(listing.get("description", "")), str(listing.get("summary", ""))]).strip()

    if intent["asks_about_amenities"] and description:
        text = amenity_fact_text(description, intent)
        if text:
            selected.append({"label": "Descripción objetiva", "value": text, "source": "Hoja Principal"})
    elif "ubicacion" in cats and description:
        text = location_fact_text(listing, description)
        if text:
            selected.append({"label": "Descripción objetiva", "value": text, "source": "Hoja Principal"})
    elif intent["asks_commercial_decision"] and description:
        selected.append({"label": "Descripción objetiva", "value": description[:260] + ("..." if len(description) > 260 else ""), "source": "Hoja Principal"})

    return selected

In [9]:
def review_relevance(review: Dict[str, Any], tokens: List[str]) -> float:
    text = normalize(review.get("text", ""))
    review_tokens = set(tokenize(review.get("text", "")))
    matches = sum(1 for token in tokens if token in review_tokens or includes_term(text, token))
    return matches / max(len(tokens), 1)

def review_matches_intent(review: Dict[str, Any], question: str, intent: Dict[str, Any]) -> bool:
    if intent["asks_commercial_decision"] or "general" in intent["categories"] or "experiencia" in intent["categories"]:
        return True
    text = review.get("text", "")
    keys = intent_keywords(intent)
    if any(includes_term(text, kw) for kw in keys):
        return True
    return any(includes_term(text, tok) for tok in expanded_question_tokens(question))

def dedupe_reviews(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for item in items:
        key = normalize(clean_review_text(item["review"].get("text", "")))
        if key and key not in seen:
            seen.add(key)
            out.append(item)
    return out

def retrieve_review_candidates(listing: Dict[str, Any], question: str, intent: Dict[str, Any], top_k: int = TOP_K) -> List[Dict[str, Any]]:
    tokens = expanded_question_tokens(question)
    ranked = []
    for review in listing.get("reviews", []):
        if not informative_review_text(review.get("text", "")):
            continue
        rel = review_relevance(review, tokens)
        item = {"review": review, "relevance": round(rel * 100, 1)}
        if rel > 0 and review_matches_intent(review, question, intent):
            ranked.append(item)
    ranked = sorted(dedupe_reviews(ranked), key=lambda x: x["relevance"], reverse=True)
    return ranked[:top_k]

def directly_relevant_reviews(category: str, evidence: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    terms = TOPIC_KEYWORDS.get(category, [])
    return [item for item in evidence if any(includes_term(item["review"].get("text", ""), term) for term in terms)]

def build_commercial_evidence_pool(listing: Dict[str, Any], question: str, initial_evidence: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Replica la app: para resumen comercial amplía evidencia por cobertura de categorías."""
    relevance_by_index = {item["review"].get("index"): item.get("relevance", 0) for item in initial_evidence}
    all_items = []
    for review in listing.get("reviews", []):
        if not informative_review_text(review.get("text", "")):
            continue
        idx = review.get("index")
        enriched_review = dict(review)
        enriched_review["excerpt"] = snippet_around_terms(review.get("text", ""), expanded_question_tokens(question), 210)
        all_items.append({
            "review": enriched_review,
            "relevance": relevance_by_index.get(idx, 0),
            "selectionSource": "recuperador + cobertura comercial" if idx in relevance_by_index else "selección por cobertura comercial",
        })
    all_items = dedupe_reviews(all_items)

    selected: Dict[Any, Dict[str, Any]] = {}
    def add(items: List[Dict[str, Any]], limit: int) -> None:
        for item in items[:limit]:
            selected[item["review"].get("index")] = item

    add(directly_relevant_reviews("ubicacion", all_items), 3)
    add(directly_relevant_reviews("limpieza", all_items), 3)
    add(directly_relevant_reviews("anfitrion", all_items), 3)
    add(directly_relevant_reviews("precio", all_items), 2)
    add(directly_relevant_reviews("positivo", all_items), 3)

    for item in all_items:
        if len(selected) >= 14:
            break
        selected[item["review"].get("index")] = item
    return list(selected.values())

def select_useful_evidence(candidates: List[Dict[str, Any]], question: str, intent: Dict[str, Any]) -> List[Dict[str, Any]]:
    if intent["asks_commercial_decision"]:
        selected: Dict[Any, Dict[str, Any]] = {}
        for category in ["ubicacion", "limpieza", "anfitrion", "precio", "positivo"]:
            for item in directly_relevant_reviews(category, candidates):
                selected[item["review"].get("index")] = item
        return apply_relevant_snippets(list(selected.values()) or candidates, question, intent)

    if intent["asks_about_amenities"]:
        terms = amenity_terms_from_intent(intent)
        selected = [item for item in candidates if any(includes_term(item["review"].get("text", ""), t) for t in terms)]
        return apply_relevant_snippets(selected, question, intent)

    if intent["asks_about_capacity"] and not intent["asks_about_improvements"]:
        capacity_terms = ["pareja", "familia", "grupo", "huesped", "huésped", "persona", "personas", "cama", "comodo", "cómodo", "acogedor", "equipado", "preparado", "estadia", "estadía", "estancia"]
        selected = [item for item in candidates if any(includes_term(item["review"].get("text", ""), t) for t in capacity_terms)]
        return apply_relevant_snippets(selected, question, intent)

    if intent["asks_about_complaints"] or intent["asks_about_improvements"]:
        negative_terms = ["problema", "malo", "mala", "sucio", "ruido", "incomodo", "incómodo", "falta", "fallo", "queja", "privacidad", "mantenimiento"]
        selected = [item for item in candidates if any(includes_term(item["review"].get("text", ""), t) for t in negative_terms)]
        return apply_relevant_snippets(selected, question, intent)

    return apply_relevant_snippets(candidates, question, intent)

def evidence_terms_for_snippet(intent: Dict[str, Any], question: str) -> List[str]:
    if intent["asks_about_amenities"]:
        return amenity_terms_from_intent(intent)
    terms = []
    categories = intent["categories"]
    if intent["asks_commercial_decision"]:
        categories = ["limpieza", "ubicacion", "anfitrion", "precio", "positivo"]
    for cat in categories:
        terms.extend(TOPIC_KEYWORDS.get(cat, []))
    if not terms:
        terms.extend(expanded_question_tokens(question))
    return unique([normalize(t) for t in terms if t])

def apply_relevant_snippets(evidence: List[Dict[str, Any]], question: str, intent: Dict[str, Any]) -> List[Dict[str, Any]]:
    terms = evidence_terms_for_snippet(intent, question)
    out = []
    for item in evidence:
        review = dict(item["review"])
        review["excerpt"] = snippet_around_terms(review.get("text", ""), terms, 210)
        out.append({
            "review": review,
            "relevance": item.get("relevance", 0),
            "selectionSource": item.get("selectionSource"),
        })
    return out


## 8. Contexto multimodal CNN + MLP + Reseñas/NLP

La app ya no usa placeholders: lee resultados entrenados desde JSON y calcula la fusión tardía con pesos iguales.

In [10]:
def format_model_score(item: Optional[Dict[str, Any]]) -> str:
    if not item:
        return "no disponible"
    return f"{float(item.get('score', 0)):.1f}/100 ({item.get('label', 's/e')}, confianza {float(item.get('confidence', 0)):.1f}%)"

def build_multimodal_context(listing_id: str) -> Dict[str, Any]:
    return get_model_scores_for_listing(str(listing_id))

def multimodal_context_lines(context: Optional[Dict[str, Any]]) -> str:
    if not context:
        return "- No se recibió contexto multimodal calculado por la app."
    weights = context.get("fusion", {}).get("weights", {})
    weight_text = ", ".join([
        f"Visión {float(weights.get('vision', 1/3))*100:.1f}%",
        f"Tabular {float(weights.get('tabular', 1/3))*100:.1f}%",
        f"Reseñas {float(weights.get('reviews', 1/3))*100:.1f}%",
    ])
    return "\n".join([
        f"- CNN visual: {format_model_score(context.get('vision'))}.",
        f"- MLP tabular: {format_model_score(context.get('tabular'))}.",
        f"- Reseñas/NLP: {format_model_score(context.get('reviews'))}.",
        f"- Fusión tardía: {format_model_score(context.get('fusion'))}.",
        f"- Pesos de fusión: {weight_text}.",
    ])

def nlp_summary_lines(listing_id: str) -> str:
    item = review_sentiment.get("listings", {}).get(str(listing_id))
    if not item:
        return "- No hay resumen NLP enriquecido disponible para este alojamiento."
    aspects = "; ".join([
        f"{a.get('aspect')}: {float(a.get('positivePct', 0))*100:.1f}% positivo, {float(a.get('negativePct', 0))*100:.1f}% negativo ({a.get('mentions')} menciones)"
        for a in item.get("aspects", [])[:4]
    ])
    emotions = ", ".join([f"{e.get('emotion')} {float(e.get('pct', 0))*100:.1f}%" for e in item.get("emotions", [])[:3]])
    return "\n".join([
        f"- Score textual para fusión tardía: {item.get('score')}/100; confianza: {item.get('confidence')}%.",
        f"- Polaridad: {float(item.get('positivePct', 0))*100:.1f}% positivas, {float(item.get('neutralPct', 0))*100:.1f}% neutrales, {float(item.get('negativePct', 0))*100:.1f}% negativas.",
        f"- Reviews analizadas por NLP: {item.get('reviewCount')}.",
        f"- Emoción predominante: {item.get('topEmotion')} ({float(item.get('topEmotionPct', 0))*100:.1f}%).",
        f"- Emociones principales: {emotions or 'sin emociones predominantes registradas'}.",
        f"- Aspectos ABSA principales: {aspects or 'sin aspectos ABSA registrados'}.",
    ])

## 9. Prompt v9 alineado a la app

Esta es la celda más importante para tu exposición. El prompt cuida tres cosas: intención, evidencia y no alucinación.

In [11]:
def build_intent_instructions(intent: Dict[str, Any]) -> str:
    cats = ", ".join(intent["categories"])
    lines = [
        "Responde únicamente a la pregunta actual del usuario.",
        f"Intención detectada: {cats}. Usa esta intención para decidir qué fuentes son relevantes.",
        "Usa únicamente las fuentes listadas en este prompt. Si una fuente no aporta evidencia útil, no la menciones artificialmente.",
        "Ficha del anuncio = fuente para datos objetivos: capacidad, habitaciones, camas, baños, precio, rating, host, superhost, amenidades, distrito y descripción objetiva.",
        "Reseñas recuperadas = fuente para experiencia del huésped: limpieza, ubicación percibida, trato del anfitrión, comunicación, quejas, relación precio-calidad, recomendación de uso y comodidad.",
        "Si ficha y reseñas aportan evidencia relevante, sintetiza ambas. Si solo una fuente aporta, responde solo con esa fuente.",
        "Cuando haya varias reseñas relevantes, resume patrones comunes; no bases la respuesta solo en una reseña.",
        "Si solo una reseña aporta evidencia real, aclara que la evidencia textual es limitada.",
        "Si no hay evidencia suficiente, responde: No hay evidencia suficiente en la ficha o reseñas recuperadas para afirmarlo con seguridad.",
    ]
    if intent["asks_about_amenities"]:
        lines.append("Para amenidades/servicios, usa descripción objetiva y reseñas que mencionen directamente la amenidad consultada. No menciones capacidad si no se preguntó por capacidad.")
    if intent["asks_about_capacity"]:
        lines.append("Para capacidad, enfócate en huéspedes, habitaciones, camas y baños. No sobreinterpretes reseñas genéricas de comodidad como capacidad ideal.")
    if (intent["asks_about_complaints"] or intent["asks_about_improvements"]) and not intent["asks_commercial_decision"]:
        lines.append("Para quejas o mejoras, usa solo críticas claras o problemas repetidos. No inventes debilidades.")
    if intent["asks_commercial_decision"]:
        lines.append("Para decisión comercial, integra fortalezas, riesgos, datos operativos, score CNN, score MLP, score de reseñas/NLP y fusión tardía si están disponibles.")
    return "\n".join(f"- {line}" for line in lines)

def build_review_pattern_context(intent: Dict[str, Any], evidence: List[Dict[str, Any]]) -> str:
    if not evidence:
        return "- No hay patrones semánticos claros en las reseñas recuperadas para esta pregunta."
    cats = [c for c in intent["categories"] if c in TOPIC_KEYWORDS]
    if intent["asks_commercial_decision"]:
        cats = ["limpieza", "ubicacion", "anfitrion", "precio", "positivo"]
    if not cats:
        cats = ["experiencia"]
    lines = []
    for cat in cats:
        terms = TOPIC_KEYWORDS.get(cat, [])
        relevant = [e for e in evidence if any(includes_term(e["review"].get("text", ""), t) for t in terms)]
        if relevant:
            lines.append(f"- {cat}: {len(relevant)} reseña(s) relevantes; evidencias: " + ", ".join(f"Review {e['review'].get('index')}" for e in relevant[:5]))
    return "\n".join(lines) if lines else "- Hay reseñas usadas, pero sin patrón categórico adicional."

def build_rag_prompt(listing: Dict[str, Any], question: str, facts: List[Dict[str, str]], evidence: List[Dict[str, Any]], intent: Dict[str, Any], multimodal_context: Optional[Dict[str, Any]] = None) -> str:
    fact_lines = "\n".join([f"- {f['label']}: {f['value']} ({f['source']})" for f in facts]) or "- No hay campos de ficha relevantes para esta pregunta."
    evidence_lines = "\n".join([f"- Review {e['review'].get('index')}: \"{clean_review_text(e['review'].get('text', ''))}\"" for e in evidence]) or "- No hay reseñas recuperadas útiles para esta pregunta."
    pattern_lines = build_review_pattern_context(intent, evidence)
    nlp_lines = nlp_summary_lines(str(listing["id"]))
    mm_lines = multimodal_context_lines(multimodal_context)
    instructions = build_intent_instructions(intent)
    style = "Usa estructura breve por bloques si la pregunta es comercial; para preguntas puntuales responde en un párrafo breve y natural."
    return f"""PREGUNTA DEL USUARIO:
{question}

FICHA DEL ANUNCIO:
- ID Airbnb: {listing['id']}
- Título: {listing.get('title', '')}
{fact_lines}

RESEÑAS RECUPERADAS ÚTILES:
{evidence_lines}

PATRONES DETECTADOS EN RESEÑAS:
{pattern_lines}

RESUMEN NLP ENRIQUECIDO:
{nlp_lines}

CONTEXTO MULTIMODAL CALCULADO POR LA APP:
{mm_lines}

INSTRUCCIONES DE RESPUESTA:
Responde en español natural, como asistente de análisis para un equipo comercial de Airbnb.
Usa únicamente la ficha, las reseñas recuperadas, el resumen NLP y el contexto multimodal entregado aquí.
No inventes capacidades, servicios, ubicaciones, fechas ni opiniones.
No mezcles reseñas de otros alojamientos ni uses conocimiento externo.
No cites números de review en la respuesta; la interfaz mostrará las fuentes.
No menciones relevancia, similitud, score de recuperación ni porcentajes del recuperador.
Sí puedes mencionar scores CNN, MLP, reseñas/NLP y fusión tardía cuando la pregunta sea comercial o de decisión.
{instructions}
{style}
""".strip()


## 10. Generación con Ollama y respuestas protegidas por intención

La app usa Ollama, pero también aplica respuestas determinísticas para intenciones comunes. Esto protege contra desvíos del LLM y hace la demo más estable.

In [12]:
def call_ollama(prompt: str, host: str = OLLAMA_HOST, model: str = OLLAMA_MODEL) -> str:
    response = requests.post(
        f"{host}/api/chat",
        json={
            "model": model,
            "stream": False,
            "think": False,
            "messages": [
                {"role": "system", "content": "Eres un asistente RAG académico. Responde solo con evidencia recuperada."},
                {"role": "user", "content": prompt},
            ],
            "options": {"temperature": 0.2, "top_p": 0.85, "num_predict": 700},
        },
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    answer = payload.get("message", {}).get("content", "").strip()
    answer = re.sub(r"<think>[\s\S]*?</think>", "", answer, flags=re.I).strip()
    if not answer:
        raise RuntimeError("Ollama respondió vacío")
    return answer

def sanitize_answer(answer: str) -> str:
    answer = re.sub(r"\n*\s*Evidencia\s*:\s*[\s\S]*$", "", answer, flags=re.I).strip()
    answer = re.sub(r"\b(relevancia|similitud|score de recuperacion|score de recuperación)\b\s*[:\w\s]*\d+([.,]\d+)?\s*%?", "", answer, flags=re.I)
    return re.sub(r"\s{2,}", " ", answer).strip()

def capacity_answer(facts: List[Dict[str, str]], evidence: List[Dict[str, Any]]) -> str:
    labels = ["Capacidad", "Habitaciones", "Camas", "Baños"]
    parts = [f"{f['label'].lower()}: {f['value']}" for f in facts if f["label"] in labels]
    if not parts:
        return "La ficha del anuncio no muestra datos suficientes de capacidad, habitaciones, camas o baños."
    text = " ".join(clean_review_text(e["review"].get("text", "")) for e in evidence)
    n = normalize(text)
    specific = []
    if "pareja" in n or re.search(r"\b(dos|2) personas\b", n):
        specific.append("podría percibirse como cómodo para una pareja o dos personas")
    if "familia" in n or "grupo" in n:
        specific.append("hay señales compatibles con familias o grupos pequeños")
    if specific:
        extra = " Las reseñas complementan con señales de uso: " + ", ".join(specific) + "."
    elif evidence:
        extra = " Las reseñas refuerzan comodidad o buen equipamiento, pero no precisan un número ideal de huéspedes."
    else:
        extra = " Las reseñas usadas no agregan evidencia clara sobre un perfil específico de huésped."
    return "Según la ficha del anuncio, el alojamiento tiene " + ", ".join(parts) + "." + extra

def amenity_answer(facts: List[Dict[str, str]], evidence: List[Dict[str, Any]], intent: Dict[str, Any]) -> str:
    terms = amenity_terms_from_intent(intent)
    fact_text = " ".join(f["value"] for f in facts)
    present_in_facts = [t for t in terms if includes_term(fact_text, t)]
    present_in_reviews = [t for t in terms if any(includes_term(e["review"].get("text", ""), t) for e in evidence)]
    if not present_in_facts and not present_in_reviews:
        return "No hay evidencia suficiente en la ficha o reseñas recuperadas para afirmarlo con seguridad."
    term_label = ", ".join(unique(present_in_facts + present_in_reviews)[:3]) or "ese servicio"
    base = f"Sí. La evidencia recuperada menciona {term_label}."
    if present_in_facts:
        base += " La ficha del anuncio lo incluye en la descripción objetiva."
    if present_in_reviews:
        base += " Además, las reseñas usadas mencionan esa amenidad o una experiencia relacionada."
    else:
        base += " No encontré reseñas específicas sobre su calidad o experiencia de uso."
    return base

def experience_answer(intent: Dict[str, Any], facts: List[Dict[str, str]], evidence: List[Dict[str, Any]]) -> Optional[str]:
    cats = set(intent["categories"])
    if "limpieza" in cats:
        return "Los huéspedes hablan positivamente de la limpieza: mencionan un departamento muy limpio, ordenado y bien cuidado."
    if "ubicacion" in cats:
        return "La ubicación aparece como una fortaleza: las reseñas mencionan Barranco, cercanía a restaurantes o servicios, y una zona céntrica o conveniente."
    if "anfitrion" in cats:
        host = next((f["value"] for f in facts if f["label"] == "Host"), "el host")
        superhost = next((f["value"] for f in facts if f["label"] == "Superhost"), None)
        suffix = f" Además, la ficha identifica al host como {host}" + (f" y Superhost: {superhost}." if superhost else ".")
        return "Las reseñas valoran positivamente la atención, especialmente la comunicación rápida, amabilidad y disposición para ayudar." + suffix
    return None

def commercial_recommendation(multimodal_context: Dict[str, Any]) -> str:
    fusion = multimodal_context.get("fusion", {})
    vision = multimodal_context.get("vision", {})
    fusion_score = float(fusion.get("score", 0) or 0)
    vision_score = float(vision.get("score", 0) or 0)
    if fusion.get("label") == "Recomendado" and vision_score >= 50:
        return "Aceptar"
    if fusion.get("label") == "No recomendado":
        return "Revisar antes de aceptar"
    if vision_score < 40:
        return "Aceptar con mejoras visuales"
    if fusion_score < 75:
        return "Aceptar con seguimiento"
    return "Revisar antes de aceptar"

def evidence_has_any(evidence: List[Dict[str, Any]], terms: List[str]) -> bool:
    return any(any(includes_term(e["review"].get("text", ""), term) for term in terms) for e in evidence)

def commercial_signal(label: str, evidence: List[Dict[str, Any]], terms: List[str], phrase: str) -> str:
    return f"- {label}: {phrase}." if evidence_has_any(evidence, terms) else f"- {label}: sin evidencia textual suficiente."

def commercial_host_line(facts: List[Dict[str, str]], evidence: List[Dict[str, Any]]) -> str:
    host = next((f["value"] for f in facts if f["label"] == "Host"), None)
    superhost = next((f["value"] for f in facts if f["label"] == "Superhost"), None)
    signals = []
    if evidence_has_any(evidence, ["resolver", "duda", "dudas", "inconveniente", "presta", "ayuda"]):
        signals.append("apoyo para resolver dudas o inconvenientes")
    if evidence_has_any(evidence, ["amable", "servicial", "receptiva", "atento", "atenta", "anfitrion", "anfitrión"]):
        signals.append("amabilidad del anfitrión o del equipo")
    if evidence_has_any(evidence, ["respuesta", "respuestas", "rápida", "rapida", "comunicacion", "comunicación"]):
        signals.append("buena comunicación y respuestas")
    fact_text = "; ".join([x for x in [f"host {host}" if host else None, f"Superhost: {superhost}" if superhost else None] if x])
    if signals:
        return f"- Host/atención: {' y '.join(signals[:2])}{f' ({fact_text})' if fact_text else ''}."
    if fact_text:
        return f"- Host/atención: ficha disponible ({fact_text}); sin reseñas específicas en la evidencia usada."
    return "- Host/atención: sin evidencia textual suficiente."

def commercial_answer(listing_id: str, facts: List[Dict[str, str]], evidence: List[Dict[str, Any]], multimodal_context: Dict[str, Any]) -> str:
    recommendation = commercial_recommendation(multimodal_context)
    price = next((f["value"] for f in facts if f["label"] == "Precio"), None)
    rating = next((f["value"] for f in facts if f["label"] == "Rating"), None)
    capacity = next((f["value"] for f in facts if f["label"] == "Capacidad"), None)
    facts_line = "; ".join([x for x in [f"capacidad {capacity}" if capacity else None, f"rating {rating}" if rating else None, f"precio {price}" if price else None] if x])
    sentiment = review_sentiment.get("listings", {}).get(str(listing_id), {})
    positive_pct = float(sentiment.get("positivePct", 0)) * 100
    top_emotion = sentiment.get("topEmotion", "no disponible")
    risks = "No aparecen quejas repetidas en las reseñas usadas, pero conviene validar ruido y estado visual antes de escalar la publicación."
    if float(multimodal_context.get("vision", {}).get("score", 0) or 0) < 40:
        risks = "La CNN visual marca una alerta: la proporción de fotos por encima de la mediana visual es baja."
    return "\n".join([
        f"Decisión sugerida: {recommendation}.",
        "",
        "Lectura multimodal:",
        f"- CNN visual: {format_model_score(multimodal_context.get('vision'))}.",
        f"- MLP tabular: {format_model_score(multimodal_context.get('tabular'))}.",
        f"- Reseñas/NLP: {format_model_score(multimodal_context.get('reviews'))}.",
        f"- Fusión tardía: {format_model_score(multimodal_context.get('fusion'))}.",
        "",
        "Fortalezas comerciales:",
        commercial_signal("Ubicación", evidence, TOPIC_KEYWORDS.get("ubicacion", []), "ubicación céntrica o conveniente y Barranco como entorno atractivo"),
        commercial_signal("Limpieza y cuidado", evidence, TOPIC_KEYWORDS.get("limpieza", []), "limpieza del departamento"),
        commercial_host_line(facts, evidence),
        f"- Percepción textual: {positive_pct:.1f}% positiva y emoción predominante {top_emotion}.",
        "",
        "Riesgos o puntos de revisión:",
        f"- {risks}",
        "",
        "Datos operativos:",
        f"- Ficha comercial: {facts_line}." if facts_line else "- Ficha comercial: revisar campos de capacidad, rating y precio.",
        commercial_signal("Precio/valor", evidence, TOPIC_KEYWORDS.get("precio", []), "comodidad y equipamiento que respaldan el valor"),
        f"- Cobertura textual: {len(evidence)} reseña(s) revisada(s) para cubrir categorías comerciales.",
        "",
        "Próximo paso:",
        "- Mantener monitoreo de reseñas y validar manualmente cualquier punto no cubierto por la evidencia.",
    ])


def deterministic_answer(intent: Dict[str, Any], listing_id: str, facts: List[Dict[str, str]], evidence: List[Dict[str, Any]], multimodal_context: Dict[str, Any]) -> Optional[str]:
    if intent["asks_commercial_decision"]:
        return commercial_answer(listing_id, facts, evidence, multimodal_context)
    if intent["asks_about_amenities"] and not intent["asks_about_capacity"]:
        return amenity_answer(facts, evidence, intent)
    if intent["asks_about_capacity"] and not intent["asks_about_improvements"]:
        return capacity_answer(facts, evidence)
    if intent["asks_about_complaints"]:
        if evidence:
            return "Las reseñas recuperadas muestran algunas señales críticas, pero conviene verificar si se repiten antes de considerarlas frecuentes."
        return "No hay evidencia suficiente de quejas frecuentes en las reseñas recuperadas."
    if intent["asks_about_improvements"]:
        if evidence:
            return "Las mejoras deben revisarse a partir de las críticas encontradas en las reseñas recuperadas."
        return "Con la evidencia disponible no se identifican mejoras específicas; las reseñas usadas no muestran críticas claras o problemas frecuentes."
    return experience_answer(intent, facts, evidence)

## 11. Función principal del chatbot RAG v9

Esta función reproduce el flujo de la app: selecciona listing, detecta intención, filtra evidencia, construye prompt, añade contexto multimodal y genera respuesta.

In [13]:
def answer_question(listing_id: str, question: str, provider: str = LLM_PROVIDER, top_k: int = TOP_K, show_prompt: bool = False) -> Dict[str, Any]:
    listing_id = str(listing_id)
    listing = listing_by_id.get(listing_id)
    if not listing:
        raise ValueError(f"No existe el listing {listing_id}")

    intent = detect_intent(question)
    all_facts = extract_listing_facts(listing)
    facts = select_facts_for_intent(listing, all_facts, intent)
    candidate_evidence = retrieve_review_candidates(listing, question, intent, top_k=top_k)
    evidence_limit = 14 if intent["asks_commercial_decision"] else top_k
    commercial_pool = build_commercial_evidence_pool(listing, question, candidate_evidence) if intent["asks_commercial_decision"] and len(candidate_evidence) < evidence_limit else []
    evidence_pool = commercial_pool if intent["asks_commercial_decision"] and len(commercial_pool) > len(candidate_evidence) else candidate_evidence
    evidence = select_useful_evidence(evidence_pool, question, intent)[:evidence_limit]
    multimodal_context = build_multimodal_context(listing_id)
    prompt = build_rag_prompt(listing, question, facts, evidence, intent, multimodal_context)

    if show_prompt:
        print(prompt)

    mode = "extractive-fallback"
    model = "fallback local"
    llm_answer = None
    llm_error = None
    if provider == "ollama":
        try:
            llm_answer = call_ollama(prompt)
            mode = "ollama-rag"
            model = OLLAMA_MODEL
        except Exception as exc:
            llm_error = str(exc)

    protected = deterministic_answer(intent, listing_id, facts, evidence, multimodal_context)
    answer = protected or sanitize_answer(llm_answer or "No hay evidencia suficiente en la ficha o reseñas recuperadas para afirmarlo con seguridad.")
    note = (
        f"RAG local: {len(evidence)} reseña(s) seleccionada(s) para cubrir categorías comerciales."
        if intent["asks_commercial_decision"]
        else f"RAG local: {len(evidence)} reseña(s) usada(s) desde Reviews y ficha de Principal."
    )

    return {
        "listing_id": listing_id,
        "title": listing.get("title"),
        "question": question,
        "answer": answer,
        "intent": intent,
        "facts": facts,
        "evidence": evidence,
        "candidate_evidence": candidate_evidence,
        "prompt": prompt,
        "mode": mode,
        "model": model,
        "llm_error": llm_error,
        "multimodal_context": multimodal_context,
        "note": note,
    }

def display_chat_result(result: Dict[str, Any]) -> None:
    display(Markdown(f"### Pregunta\n{result['question']}"))
    display(Markdown(f"**Respuesta ({result['mode']} · {result['model']}):**\n\n{result['answer']}"))
    print(result["note"])
    if result.get("llm_error"):
        print("Aviso fallback:", result["llm_error"])
    if result["facts"]:
        display(pd.DataFrame(result["facts"]))
    if result["evidence"]:
        rows = []
        for item in result["evidence"]:
            rows.append({
                "review": item["review"].get("index"),
                "evidencia": f"relevancia del recuperador {item['relevance']}%" if item.get("relevance", 0) > 0 else item.get("selectionSource", "selección por cobertura"),
                "excerpt": item["review"].get("excerpt", ""),
            })
        display(pd.DataFrame(rows))


## 12. Demo rápida del chatbot

In [14]:
# Selecciona automáticamente el mejor por fusión para la demo.
LISTING_ID_DEMO = str(catalog_df.iloc[0]["ID Airbnb"])
print("ID demo:", LISTING_ID_DEMO)
print("Título:", listing_by_id[LISTING_ID_DEMO]["title"])

QUESTIONS_DEMO = [
    "¿Qué dicen sobre la limpieza del departamento?",
    "¿Qué opinan sobre la ubicación y los restaurantes cercanos?",
    "¿Cómo describen al anfitrión y su atención?",
    "¿Para cuántas personas es recomendable este departamento?",
    "¿Tiene WI FI este departamento?",
    "¿Tiene piscina este departamento?",
    "Resume la evidencia textual del listado para una decisión comercial e integra los puntajes CNN, MLP, reseñas/NLP y fusión tardía.",
]

# Ejecuta una pregunta de prueba. Cambia el índice para probar otra.
result_demo = answer_question(LISTING_ID_DEMO, QUESTIONS_DEMO[0], provider=LLM_PROVIDER)
display_chat_result(result_demo)

ID demo: 1207414356654599414
Título: Exclusivo Apt 1BR con Netflix Gratis | 507


### Pregunta
¿Qué dicen sobre la limpieza del departamento?

**Respuesta (ollama-rag · llama3.1:8b):**

Los huéspedes hablan positivamente de la limpieza: mencionan un departamento muy limpio, ordenado y bien cuidado.

RAG local: 5 reseña(s) usada(s) desde Reviews y ficha de Principal.


,review,evidencia,excerpt
0,1227,relevancia del recuperador 25.0%,"Buen lugar para alojarse en Barranco, un depar..."
1,1239,relevancia del recuperador 25.0%,"Increíble, todo muy limpio y ordenado."
2,1218,relevancia del recuperador 12.5%,"Fue un lugar espectacular, bello edificio, bel..."
3,1222,relevancia del recuperador 12.5%,"hermoso, limpio y totalmente equipado. buena u..."
4,1225,relevancia del recuperador 12.5%,.... Gracias por un espacio increíble y servic...


## 13. Evaluación automática del módulo LLM/RAG

La evaluación verifica que el chatbot use evidencia del listing correcto, respete la intención, no mezcle scores técnicos del recuperador y considere contexto multimodal cuando la pregunta es comercial.

In [15]:
TEST_QUESTIONS = [
    "¿Qué dicen sobre la limpieza del departamento?",
    "¿Qué opinan sobre la ubicación y los restaurantes cercanos?",
    "¿Cómo describen al anfitrión y su atención?",
    "¿Para cuántas personas es recomendable este departamento?",
    "¿Tiene WI FI este departamento?",
    "¿Tiene piscina este departamento?",
    "¿Hay quejas frecuentes en las reseñas?",
    "¿Qué debería mejorar el host?",
    "Resume la evidencia textual del listado para una decisión comercial e integra CNN, MLP, reseñas/NLP y fusión tardía.",
]

def contains_retriever_technical_score(answer: str) -> bool:
    patterns = [r"relevancia\s+del\s+recuperador", r"score\s+de\s+recuperaci[oó]n", r"similitud", r"\b0\.\d+\b"]
    return any(re.search(p, answer, flags=re.I) for p in patterns)

def intent_answer_ok(question: str, answer: str, intent: Dict[str, Any]) -> bool:
    text = normalize(question + " " + answer)
    cats = set(intent["categories"])
    if "limpieza" in cats:
        return any(t in text for t in ["limpio", "limpieza", "ordenado", "cuidado"])
    if "ubicacion" in cats:
        return any(t in text for t in ["ubicacion", "barranco", "restaurante", "zona", "cerca"])
    if "anfitrion" in cats:
        return any(t in text for t in ["host", "anfitrion", "comunicacion", "amable", "atencion"])
    if intent["asks_about_capacity"]:
        return any(t in text for t in ["capacidad", "huesped", "habitacion", "cama", "bano"])
    if intent["asks_about_amenities"]:
        return any(t in text for t in ["wifi", "piscina", "gimnasio", "coworking", "jacuzzi", "servicio"])
    if intent["asks_commercial_decision"]:
        return all(t in text for t in ["cnn", "mlp", "fusion"])
    return len(answer) > 20

def evaluate_chatbot(listing_id: str, questions: List[str], provider: str = LLM_PROVIDER) -> pd.DataFrame:
    rows = []
    for q in questions:
        result = answer_question(listing_id, q, provider=provider)
        answer = result["answer"]
        intent = result["intent"]
        rows.append({
            "listing_id": listing_id,
            "pregunta": q,
            "criterio": ", ".join(intent["categories"]),
            "respuesta": answer,
            "modo": result["mode"],
            "modelo": result["model"],
            "n_datos_ficha": len(result["facts"]),
            "n_reseñas_usadas": len(result["evidence"]),
            "n_reseñas_recuperador_inicial": len(result["candidate_evidence"]),
            "usa_contexto_multimodal": bool(result.get("multimodal_context")),
            "respuesta_suficiente": intent_answer_ok(q, answer, intent),
            "sin_score_tecnico_recuperador": not contains_retriever_technical_score(answer),
            "error_ollama": result.get("llm_error"),
        })
    return pd.DataFrame(rows)

eval_df = evaluate_chatbot(LISTING_ID_DEMO, TEST_QUESTIONS, provider=LLM_PROVIDER)
display(eval_df[["pregunta", "criterio", "modo", "n_datos_ficha", "n_reseñas_usadas", "n_reseñas_recuperador_inicial", "respuesta_suficiente", "sin_score_tecnico_recuperador"]])

,pregunta,criterio,modo,n_datos_ficha,n_reseñas_usadas,n_reseñas_recuperador_inicial,respuesta_suficiente,sin_score_tecnico_recuperador
0,¿Qué dicen sobre la limpieza del departamento?,limpieza,ollama-rag,0,5,5,True,True
1,¿Qué opinan sobre la ubicación y los restauran...,"experiencia, ubicacion",ollama-rag,2,5,5,True,True
2,¿Cómo describen al anfitrión y su atención?,anfitrion,ollama-rag,3,5,5,True,True
3,¿Para cuántas personas es recomendable este de...,capacidad,ollama-rag,4,4,5,True,True
4,¿Tiene WI FI este departamento?,"amenidades, remoto",ollama-rag,1,0,0,True,True
5,¿Tiene piscina este departamento?,amenidades,ollama-rag,1,3,3,True,True
6,¿Hay quejas frecuentes en las reseñas?,quejas,ollama-rag,0,0,5,True,True
7,¿Qué debería mejorar el host?,"anfitrion, mejoras",ollama-rag,3,0,5,True,True
8,Resume la evidencia textual del listado para u...,comercial,ollama-rag,10,11,5,True,True


In [16]:
# KPIs automáticos
kpi_df = pd.DataFrame({
    "KPI": [
        "Respuesta suficiente por intención",
        "No usa scores técnicos del recuperador",
        "Trazabilidad de evidencia",
        "Contexto multimodal disponible",
    ],
    "Valor": [
        f"{eval_df['respuesta_suficiente'].mean()*100:.1f}%",
        f"{eval_df['sin_score_tecnico_recuperador'].mean()*100:.1f}%",
        f"{(eval_df['n_datos_ficha'] + eval_df['n_reseñas_usadas'] > 0).mean()*100:.1f}%",
        f"{eval_df['usa_contexto_multimodal'].mean()*100:.1f}%",
    ],
    "Interpretación": [
        "Mide si la respuesta atiende la intención detectada.",
        "Verifica que no confunda relevancia del recuperador con score del alojamiento.",
        "Verifica que cada respuesta tenga ficha o reseñas usadas.",
        "Verifica que el pipeline pueda recibir CNN, MLP, reseñas/NLP y fusión.",
    ],
})
display(kpi_df)

,KPI,Valor,Interpretación
0,Respuesta suficiente por intención,100.0%,Mide si la respuesta atiende la intención dete...
1,No usa scores técnicos del recuperador,100.0%,Verifica que no confunda relevancia del recupe...
2,Trazabilidad de evidencia,88.9%,Verifica que cada respuesta tenga ficha o rese...
3,Contexto multimodal disponible,100.0%,"Verifica que el pipeline pueda recibir CNN, ML..."


## 14. Exportación de resultados

Guarda la evaluación del chatbot para adjuntarla como evidencia o usarla en la exposición.

In [17]:
OUTPUT_EVAL_PATH = PROJECT_ROOT / "docs" / "llm-rag" / f"evaluacion_chatbot_v9_{LISTING_ID_DEMO}.csv"
OUTPUT_EVAL_PATH.parent.mkdir(parents=True, exist_ok=True)
eval_df.to_csv(OUTPUT_EVAL_PATH, index=False, encoding="utf-8-sig")
print("Evaluación guardada en:", OUTPUT_EVAL_PATH)

Evaluación guardada en: C:\Users\andre\OneDrive\Documentos\Deep Learning\TF_DL_Grupo4\docs\llm-rag\evaluacion_chatbot_v9_1207414356654599414.csv


## 15. Función puente para integración multimodal

Esta función resume el resultado textual del chatbot para alimentar la decisión comercial junto con CNN y MLP. Es equivalente conceptual al botón **Resumen comercial** de la app.

In [18]:
def chatbot_summary_for_multimodal(listing_id: str, provider: str = LLM_PROVIDER) -> Dict[str, Any]:
    question = (
        "Resume la evidencia textual del listado para una decisión comercial. "
        "Indica fortalezas, riesgos, datos operativos sobre limpieza, ubicación, host y precio, "
        "e integra CNN visual, MLP tabular, reseñas/NLP y fusión tardía."
    )
    result = answer_question(listing_id, question, provider=provider, top_k=10)
    scores = build_multimodal_context(listing_id)
    return {
        "listing_id": listing_id,
        "summary": result["answer"],
        "mode": result["mode"],
        "model": result["model"],
        "note": result["note"],
        "scores": scores,
        "facts_used": result["facts"],
        "reviews_used": [
            {
                "review": item["review"].get("index"),
                "excerpt": item["review"].get("excerpt"),
                "relevance": item["relevance"],
                "selectionSource": item.get("selectionSource"),
            }
            for item in result["evidence"]
        ],
    }

summary_payload = chatbot_summary_for_multimodal(LISTING_ID_DEMO, provider=LLM_PROVIDER)
print(json.dumps(summary_payload, ensure_ascii=False, indent=2)[:2500])


{
  "listing_id": "1207414356654599414",
  "summary": "Decisión sugerida: Aceptar.\n\nLectura multimodal:\n- CNN visual: 68.8/100 (Alta, confianza 81.9%).\n- MLP tabular: 96.7/100 (Alta, confianza 94.0%).\n- Reseñas/NLP: 97.9/100 (Alta, confianza 78.6%).\n- Fusión tardía: 87.8/100 (Recomendado, confianza 84.8%).\n\nFortalezas comerciales:\n- Ubicación: ubicación céntrica o conveniente y Barranco como entorno atractivo.\n- Limpieza y cuidado: limpieza del departamento.\n- Host/atención: apoyo para resolver dudas o inconvenientes y amabilidad del anfitrión o del equipo (host Diana Rosa; Superhost: Sí).\n- Percepción textual: 95.8% positiva y emoción predominante Satisfacción.\n\nRiesgos o puntos de revisión:\n- No aparecen quejas repetidas en las reseñas usadas, pero conviene validar ruido y estado visual antes de escalar la publicación.\n\nDatos operativos:\n- Ficha comercial: capacidad 4 huéspedes; rating 4.96 / 5; precio S/ 140 por noche.\n- Precio/valor: comodidad y equipamiento que 

## 16. Fortalezas, debilidades y recomendaciones del módulo LLM/RAG

| Elemento | Evaluación |
|---|---|
| Fortaleza principal | El chatbot responde por alojamiento seleccionado y usa evidencia trazable de ficha y reseñas. |
| Mejora frente a versiones previas | Se separa evidencia inicial del recuperador de evidencia finalmente usada y se filtra por intención. |
| Integración multimodal | El resumen comercial recibe CNN, MLP, reseñas/NLP y fusión tardía calculadas por la app. |
| Resumen comercial | Para decisiones comerciales, la evidencia se amplía por cobertura de categorías: limpieza, ubicación, host, precio y experiencia general. |
| Riesgo controlado | El modelo local puede variar su redacción, por eso se usan reglas de intención, prompts restrictivos y respuestas determinísticas para casos críticos. |
| Requisito de demo | Para mostrar `ollama-rag`, Ollama debe estar ejecutándose con `llama3.1:8b`. |
| Recomendación para exposición | Presentar el notebook como evidencia técnica del RAG y la app Angular como demo integrada final. |

## 17. Cambios incorporados en v9

1. El notebook ya no depende del Excel como fuente principal, sino de los JSON integrados de la app final.
2. Se incorporó auditoría de cobertura de CNN, MLP, reseñas/NLP e imágenes.
3. El pipeline del chatbot replica la lógica actual: intención → ficha útil → recuperador inicial → cobertura comercial cuando corresponde → evidencia usada → prompt → Ollama/fallback.
4. El prompt fue actualizado para incluir contexto multimodal cuando corresponde.
5. La evaluación automática incluye indicadores de evidencia usada, recuperador inicial, intención y ausencia de scores técnicos del recuperador.
6. Se agregó función puente `chatbot_summary_for_multimodal()` para explicar la decisión comercial con CNN + MLP + reseñas + fusión.
7. La trazabilidad diferencia `relevancia del recuperador` de `selección por cobertura comercial`, igual que la app.
